# Insurance Linear Regression

Predicting medical insurance **charges** from age, BMI, number of children and smoking status using a multiple linear regression model.

**Steps covered**
1. Import libraries
2. Load `insuranceData.csv`
3. Encode categorical data with `LabelEncoder()`
4. Encode categorical data with `OneHotEncoder()`
5. Split into training and test sets (test size = 15%)
6. Train and evaluate a linear regression model

## 1. Import the libraries

In [ ]:
# numpy  - numerical arrays and maths
# pandas - loading and handling tabular data (DataFrames)
# matplotlib - plotting the results
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn tools for preprocessing, splitting, modelling and scoring
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2. Load the dataset

In [ ]:
# Read the CSV file into a pandas DataFrame
# (the CSV must be in the same folder as this notebook)
dataset = pd.read_csv('insuranceData.csv')

# Show the first five rows to check the data loaded correctly
dataset.head()

In [ ]:
# Basic information: number of rows/columns, data types and missing values
print('Shape:', dataset.shape)
print()
dataset.info()
print()
print('Missing values per column:')
print(dataset.isnull().sum())

In [ ]:
# Summary statistics for the numeric columns
dataset.describe()

## 3. Separate the features (X) and the target (y)

- **X** (independent variables): `age`, `bmi`, `children`, `smoker`
- **y** (dependent variable): `charges` — what we want to predict

In [ ]:
# iloc[:, :-1] -> every row, every column except the last one (the features)
# iloc[:, -1]  -> every row, only the last column (charges)
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

# Column positions in X: 0 = age, 1 = bmi, 2 = children, 3 = smoker
print(X[:5])

## 4. Apply `LabelEncoder()` to the categorical data

`smoker` is the only categorical column. A regression model needs numbers, so `LabelEncoder` converts each text label into an integer: **no → 0**, **yes → 1** (labels are numbered alphabetically).

In [ ]:
# Create the LabelEncoder object
label_encoder = LabelEncoder()

# fit_transform learns the unique labels ('no', 'yes') and replaces them with 0 and 1
# Column index 3 is the 'smoker' column
X[:, 3] = label_encoder.fit_transform(X[:, 3])

# Show which number each original label was mapped to
print('Label mapping:', dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_).tolist())))
print(X[:5])

## 5. Apply `OneHotEncoder()` to the categorical data

One-hot encoding turns the smoker column into separate 0/1 *dummy* columns (one per category), so the model does not read the codes as an ordered scale.

`drop='first'` removes one of the dummy columns to avoid the **dummy variable trap** (perfectly correlated columns). With two categories this leaves a single `smoker_yes` column.

In [ ]:
# ColumnTransformer applies the OneHotEncoder only to column 3 (smoker)
# remainder='passthrough' keeps the other columns (age, bmi, children) unchanged
# The encoded columns are placed FIRST in the output, followed by the passthrough columns
column_transformer = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(drop='first'), [3])],
    remainder='passthrough'
)

# Fit the encoder and transform X; convert to float so the model receives numeric data
X = np.array(column_transformer.fit_transform(X), dtype=float)

# New column order: smoker_yes, age, bmi, children
feature_names = ['smoker_yes', 'age', 'bmi', 'children']
print(pd.DataFrame(X[:5], columns=feature_names))

## 6. Split the data into a training set and a test set (test size = 15%)

In [ ]:
# test_size=0.15 -> 15% of rows are held back for testing, 85% used for training
# random_state=0 -> makes the split reproducible (same split every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=0
)

print('Training set:', X_train.shape[0], 'rows')
print('Test set:    ', X_test.shape[0], 'rows')

## 7. Train the multiple linear regression model

In [ ]:
# Create the linear regression model and fit it to the training data
# The model learns one coefficient per feature plus an intercept
regressor = LinearRegression()
regressor.fit(X_train, y_train)

# Display the learned intercept and coefficients
print('Intercept:', round(regressor.intercept_, 2))
coefficients = pd.DataFrame({'Feature': feature_names, 'Coefficient': regressor.coef_.round(2)})
coefficients

## 8. Predict the test set results

In [ ]:
# Use the trained model to predict charges for the unseen test data
y_pred = regressor.predict(X_test)

# Show predicted vs actual charges side by side (first 10 rows)
np.set_printoptions(precision=2, suppress=True)
comparison = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}).round(2)
comparison.head(10)

## 9. Evaluate the model

In [ ]:
# R² score: proportion of variance in charges explained by the model (1.0 = perfect)
r2 = r2_score(y_test, y_pred)

# Mean Absolute Error: average size of the prediction error, in dollars
mae = mean_absolute_error(y_test, y_pred)

# Root Mean Squared Error: like MAE but penalises large errors more heavily
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R² score: {r2:.4f}')
print(f'MAE:      {mae:,.2f}')
print(f'RMSE:     {rmse:,.2f}')

In [ ]:
# Scatter plot of actual vs predicted charges
# Points close to the dashed red line are accurate predictions
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k')
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(limits, limits, 'r--', label='Perfect prediction')
plt.xlabel('Actual charges')
plt.ylabel('Predicted charges')
plt.title('Insurance Linear Regression: Actual vs Predicted')
plt.legend()
plt.show()

## 10. Predict charges for a new person (example)

The input must be transformed with the same encoders used on the training data.

In [ ]:
# Example: a 30-year-old smoker with a BMI of 28.5 and 1 child
# Order matches the original columns: age, bmi, children, smoker
new_person = np.array([[30, 28.5, 1, 'yes']], dtype=object)

# Apply the same LabelEncoder and OneHotEncoder that were fitted earlier
new_person[:, 3] = label_encoder.transform(new_person[:, 3])
new_person = np.array(column_transformer.transform(new_person), dtype=float)

print(f'Predicted charges: {regressor.predict(new_person)[0]:,.2f}')